# Inspect a training run

Reads the files `run_training.py` writes into an experiment folder and plots them. **No training happens here**; it is safe
to run while a job is still writing (rerun the cells to refresh).

| File | Shown as |
|---|---|
| `dataset_report.json` | per-dataset table: splits, selected samples, sampling share, fps / gap |
| `train_metrics.csv` | total loss, every loss term, per-dataset losses, learning rate, gradient norm, GPU memory, throughput, sampling shares |
| `val_metrics.csv` | validation curves per dataset (validation split only) |
| `summary.json` | final status and dataset contribution (when the run has ended) |

In [ ]:
EXP = "../runs/smoke_1pct"   # experiment folder (relative to this notebook, or absolute)

import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

p = lambda f: os.path.join(EXP, f)
assert os.path.isdir(EXP), f"{EXP} not found"
print("files:", sorted(os.listdir(EXP)))
train = pd.read_csv(p("train_metrics.csv")) if os.path.exists(p("train_metrics.csv")) else None
val = pd.read_csv(p("val_metrics.csv")) if os.path.exists(p("val_metrics.csv")) else None
report = json.load(open(p("dataset_report.json"))) if os.path.exists(p("dataset_report.json")) else None
summary = json.load(open(p("summary.json"))) if os.path.exists(p("summary.json")) else None
names = [d["name"] for d in report["datasets"]] if report else []
print(f"train rows: {0 if train is None else len(train)}, val rows: {0 if val is None else len(val)}, datasets: {names}")

## Datasets used

In [ ]:
if report:
    rows = []
    for d in report["datasets"]:
        c = d["converted"]
        rows.append({"dataset": d["title"],
                     "train seq / val seq / test seq": f"{c['train']['sequences']} / {c['val']['sequences']} / {c['test']['sequences']}",
                     "validation": d["validation"]["mode"], "excluded": len(d["excluded_train"]) + len(d["excluded_val"]),
                     "eligible samples": d["eligible_samples"], "fraction": d["fraction_requested"],
                     "selected samples": d["selected_samples"], "level": d["selection_level"],
                     "weight": d["weight"], "share of draws": round(d["dataset_probability"], 3),
                     "draws per sample": round(d["expected_draws_per_selected_sample"], 1),
                     "fps": "%.0f-%.0f" % tuple(d["fps"]), "hidden frames": "%d-%d" % tuple(d["hidden_frames"]),
                     "gap ms": "%.1f-%.1f" % tuple(d["gap_ms"])})
    display(pd.DataFrame(rows).set_index("dataset"))
    print(f"global batch {report['global_batch']} ({report['batch_per_gpu']} x {report['gpus']} GPU), {report['steps']} steps")

## Total loss and loss components

In [ ]:
BASE = {"step", "epoch", "samples_seen", "lr", "loss", "grad_norm", "step_time_s", "samples_per_s", "gpu_mem_gb",
        "c", "r", "k", "nu", "R_us"}
per_ds = lambda col: any(col.startswith(n + "_") for n in names)
terms = [c for c in train.columns if c not in BASE and not per_ds(c)] if train is not None else []
if train is not None and len(train):
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(train.step, train.loss)
    ax[0].set_title("total loss (weighted sum)"); ax[0].set_xlabel("step")
    for t in terms:
        if train[t].notna().any():
            ax[1].plot(train.step, train[t], label=t)
        else:
            print(f"{t}: N/A in every logged interval (e.g. photo without hidden frames)")
    ax[1].set_title("loss terms (unweighted; see config loss: for weights)"); ax[1].set_xlabel("step"); ax[1].legend()
    plt.tight_layout(); plt.show()

## Loss terms per dataset
Exact split of the batch terms that are means over samples. `sigreg` is a statistic of the whole batch and has no per-dataset value. Gaps in a line are N/A (no such samples in that interval), not zeros.

In [ ]:
if train is not None and len(train):
    pt = sorted({c[len(n) + 1:] for c in train.columns for n in names if c.startswith(n + "_")}
                - {"samples", "share_observed", "share_target"})
    fig, ax = plt.subplots(1, len(pt), figsize=(5 * len(pt), 3.6), squeeze=False)
    for a, t in zip(ax[0], pt):
        for i, n in enumerate(names):
            col = f"{n}_{t}"
            if col in train and train[col].notna().any():
                a.plot(train.step, train[col], label=n, color=f"C{i}")
        a.set_title(t); a.set_xlabel("step"); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Validation per dataset (validation split, never the test split)

In [ ]:
if val is not None and len(val):
    metrics = [c for c in val.columns if c not in ("step", "dataset", "samples")]
    fig, ax = plt.subplots(1, len(metrics), figsize=(4.5 * len(metrics), 3.6), squeeze=False)
    for a, m in zip(ax[0], metrics):
        for n, g in val.groupby("dataset"):
            a.plot(g.step, g[m], marker="o", ms=3, label=n, color=f"C{names.index(n)}" if n in names else None)
        a.set_title(m); a.set_xlabel("step"); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(val.groupby("dataset").tail(1).set_index("dataset"))
else:
    print("no validation rows yet")

## Dataset sampling: observed vs target share

In [ ]:
if train is not None and len(train):
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
    for i, n in enumerate(names):
        ax[0].plot(train.step, train[f"{n}_share_observed"], label=f"{n} observed", color=f"C{i}")
        ax[0].axhline(train[f"{n}_share_target"].iloc[-1], color=f"C{i}", ls="--", lw=0.8)
        ax[1].bar(n, train[f"{n}_samples"].sum(), color=f"C{i}")
    ax[0].set_title("cumulative share of samples (dashed: target)"); ax[0].set_xlabel("step"); ax[0].legend(fontsize=8)
    ax[1].set_title("training samples seen per dataset (logged intervals)")
    plt.tight_layout(); plt.show()

## Learning rate, gradient norm, GPU memory, throughput

In [ ]:
if train is not None and len(train):
    fig, ax = plt.subplots(1, 4, figsize=(20, 3.5))
    for a, (col, title) in zip(ax, [("lr", "learning rate"), ("grad_norm", "gradient norm (before clipping)"),
                                    ("gpu_mem_gb", "peak GPU memory (GB)"), ("samples_per_s", "samples / s")]):
        a.plot(train.step, train[col]); a.set_title(title); a.set_xlabel("step")
    plt.tight_layout(); plt.show()
    last = train.iloc[-1]
    print(f"last row: step {int(last.step)}, {int(last.samples_seen)} samples seen, {last.step_time_s:.2f} s/step, "
          f"sensor c {last.c:.3f} r {last.r:.3f} k {last.k:.2f} nu {last.nu:.3f} R {last.R_us:.0f} us")

## Summary

In [ ]:
if summary:
    print(f"status {summary['status']} at step {summary.get('step')} of {summary.get('steps_planned')}, "
          f"{summary.get('samples_seen')} samples seen; best: {summary.get('best')}")
    display(pd.DataFrame(summary.get("contribution", {})).T)
else:
    print("no summary.json yet (the run is still going or was killed before writing it)")